# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1: Get Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
if Path(MANIFEST_CSV).exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame({"handle": list_all_creators, "found": False})

# Load or initialize consolidated creator data pulled so far
if Path(CONSOLIDATED_CSV).exists():
    df_creators = pd.read_csv(CONSOLIDATED_CSV)
else:
    df_creators = pd.DataFrame()

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

# check handles to find
handles_to_find = manifest.loc[~manifest["found"], "handle"].tolist()
print(f"{len(handles_to_find)} handles left to find out of {len(manifest)} total.\n")

21284 handles left to find out of 21598 total.



In [6]:
manifest['found'].sum()

np.int64(314)

In [6]:
# try first 100 handles first
top_n = 4000
handles_to_find = handles_to_find[:top_n]

In [6]:
# Phase 1: chunk_size=5, single pass through everything not yet found
still_not_found, found_usernames, df_creators = run_pass(handles_to_find, chunk_size=5, df_creators=df_creators)

manifest["found"] = manifest["handle"].isin(found_usernames)
manifest.to_csv(MANIFEST_CSV, index=False)
print(f"\nPhase 1 done. {len(still_not_found)} handles still not found.\n")

[chunk_size=5] 1/20: searching ['yourtitodrew', 'docalvinfrancisco', 'komadronarsdonnamae', 'mommycris11', 'erickapineda09']
Rate limited (attempt 1/5). Waiting 5.8s...
Rate limited (attempt 2/5). Waiting 10.4s...
[chunk_size=5] 2/20: searching ['enverparaisoflorendo', 'lunakrizzle', '123456789diosa', 'itsmetrebbb', 'chrishannaustria']
Rate limited (attempt 1/5). Waiting 5.7s...
Rate limited (attempt 2/5). Waiting 10.1s...
Rate limited (attempt 3/5). Waiting 20.6s...
[chunk_size=5] 3/20: searching ['majeldaily', 'nursemidwifedonna', 'mommyfebb_', 'rena_dcflorida', 'johnbenedictabell3']
Rate limited (attempt 1/5). Waiting 5.3s...
Rate limited (attempt 2/5). Waiting 10.7s...
[chunk_size=5] 4/20: searching ['christian_hedriana', 'shashabudolera', 'fitdadlex', 'florence_delacruz', 'everygirl88']
Rate limited (attempt 1/5). Waiting 5.9s...
Rate limited (attempt 2/5). Waiting 11.0s...
Rate limited (attempt 3/5). Waiting 20.9s...
[chunk_size=5] 5/20: searching ['giannedlontocmd', 'mariavickyc

In [18]:
# Phase 2: chunk_size=1, only for leftovers from phase 1
if still_not_found:
    still_not_found, found_usernames, df_creators = run_pass(still_not_found, chunk_size=1, df_creators=df_creators)

    found_usernames = set(df_creators["username"])
    manifest["found"] = manifest["handle"].isin(found_usernames)
    manifest.to_csv(MANIFEST_CSV, index=False)

print(f"\nDone. {int(manifest['found'].sum())} / {len(manifest)} handles found.")

[chunk_size=1] 1/40: searching ['yourtitodrew']
[chunk_size=1] 2/40: searching ['itsmetrebbb']
Rate limited (attempt 1/5). Waiting 5.4s...
Rate limited (attempt 2/5). Waiting 10.4s...
Rate limited (attempt 3/5). Waiting 20.8s...
Rate limited (attempt 4/5). Waiting 40.2s...
[chunk_size=1] 3/40: searching ['majeldaily']
[chunk_size=1] 4/40: searching ['christian_hedriana']
[chunk_size=1] 5/40: searching ['giannedlontocmd']
Rate limited (attempt 1/5). Waiting 5.8s...
Rate limited (attempt 2/5). Waiting 11.0s...
Rate limited (attempt 3/5). Waiting 20.2s...
[chunk_size=1] 6/40: searching ['mariavickychan']
[chunk_size=1] 7/40: searching ['chebureche0430']
[chunk_size=1] 8/40: searching ['nicoleee062606']
Rate limited (attempt 1/5). Waiting 5.0s...
Rate limited (attempt 2/5). Waiting 10.2s...
Rate limited (attempt 3/5). Waiting 20.8s...
Rate limited (attempt 4/5). Waiting 40.5s...
  ⚠️  Still rate-limited after 5 attempts — giving up on this chunk for now.
  ⚠️  Search failed: {'code': 36009

ValueError: too many values to unpack (expected 2)

# Step 2: Extract List of existing target collaborations to avoid invitation conflicts

In [8]:
creator_open_id_list = df_creators.loc[df_creators['username'].isin(set(manifest['handle'])), 'creator_open_id'].tolist()
product_id_list = ['1734810555690551128']

# find all open IDs with conflicts
all_conflict_items = []

for i in range(0, len(creator_open_id_list), 50):
    batch = creator_open_id_list[i:i + 50]
    print(f"Checking batch {i // 50 + 1} ({len(batch)} creators)...")

    result = check_target_collaboration_conflicts(
        creator_open_id_list=batch,
        product_id_list=product_id_list,
    )

    if result.get("code") == 0:
        all_conflict_items.extend(result["data"]["conflict_items"])
    else:
        print(f"  ⚠️  Batch failed: {result}")

df_all_collab_conflicts = pd.DataFrame(all_conflict_items)

Checking batch 1 (50 creators)...
Checking batch 2 (50 creators)...
Checking batch 3 (50 creators)...
Checking batch 4 (50 creators)...
Checking batch 5 (50 creators)...
Checking batch 6 (50 creators)...
Checking batch 7 (14 creators)...


## Prepare shortlisted file without conflicts for batch processing

In [29]:
# merge all extracted data
df_creators_list_id_conflict = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_all_collab_conflicts, how='left', on="creator_open_id")

In [30]:
# set aside all new creators with open IDs but no conflicts
df_creators_list_id_new = df_creators_list_id_conflict[df_creators_list_id_conflict['creator_open_id'].notnull() & df_creators_list_id_conflict['existing_collaboration_id'].isnull()].copy()

In [33]:
# check counts
num_creators_with_conflict = df_creators_list_id_conflict['existing_collaboration_id'].notnull().sum()
num_creators_with_id = df_creators_list_id_conflict['creator_open_id'].notnull().sum()

print(f"{num_creators_with_conflict} out of {num_creators_with_id} with conflicts. {df_creators_list_id_new.shape[0]} new creator_open_id remaning for new target collaborations")

81 out of 314 with conflicts. 233 new creator_open_id remaning for new target collaborations


# Step 3: Create new Target Collaboration in batches of 50 new creators

In [34]:
# Set Default Values
message = "Hi {{user_name}}! \n\nWe'd love to have you as a Vitami affiliate. We make PMS Relief Gummies to help women get through their period with less discomfort. You'll get 20% commission plus a free sample for 1 TikTok video. Kindly accept this invite and request your sample if you're interested so we can ship it right away. Hoping to spread the word on better period care together! \u200d\u200d"
end_time = 2101132799
products = [{
    "id": '1734810555690551128',
    "target_commission_rate":2000, 
    "shop_ads_commission_rate": 300
}]
seller_contact_info = {
    "email": 'admin@vitamigummies.com'
}
free_sample_rule = {
    'has_free_sample': True,
    'is_sample_approval_exempt': True
}

In [ ]:
# # Batch-create target collaborations: group by batch_name, then chunk each group's creators into groups of 50 (the API's max per invitation)
# results = []
 
# for batch_name, group in df_creators_list_id_new.groupby('batch_name'):
#     creator_ids = group['creator_open_id'].tolist()
#     chunks = [creator_ids[i:i + 50] for i in range(0, len(creator_ids), 50)]
 
#     for chunk_count, chunk in enumerate(chunks, start=1):
#         name = f"{batch_name}_{chunk_count:02d}"
#         print(f"Creating '{name}' with {len(chunk)} creators...")
 
#         result = create_target_collaboration(
#             name=name,
#             end_time=end_time,
#             products=products,
#             creator_user_open_ids=chunk,
#             seller_contact_info=seller_contact_info,
#             free_sample_rule=free_sample_rule,
#             message=message,
#         )
 
#         if result.get("code") == 0:
#             collab_id = result["data"]["target_collaboration"]["id"]
#             print(f"  ✅ Created: {collab_id}")
#         else:
#             collab_id = None
#             print(f"  ⚠️  Failed: {result}")
 
#         results.append({
#             "name": name,
#             "batch_name": batch_name,
#             "chunk_count": chunk_count,
#             "num_creators": len(chunk),
#             "creator_open_ids": chunk,
#             "target_collaboration_id": collab_id,
#             "code": result.get("code"),
#             "message": result.get("message"),
#         })
 
# df_target_collaborations = pd.DataFrame(results)